# DCOPF Example 2 - No Branch Thermal Limits
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

> **Note (e2 vs e1):** Branch thermal limits (`branchLimits`) are removed in this example. The model solves a pure economic dispatch with DC power flow equations only.

In [ ]:
from pyomo.environ import *
import time

# ─────────────────────────────────────────────────────────────────
# System Data
# ─────────────────────────────────────────────────────────────────
BaseMW = 100
refBus = 3

buses = [1, 2, 3]

# Generators: {index: (bus, Pgmin MW, Pgmax MW, cost $/MWh)}
gen_data = {
    1: {'bus': 1, 'Pgmin': 20, 'Pgmax': 70, 'c': 10},
    2: {'bus': 3, 'Pgmin': 40, 'Pgmax': 90, 'c': 20},
}

# Loads: {bus: MW}
load_data = {1: 0, 2: 100, 3: 0}

# Branches: {index: (from_bus, to_bus, reactance x pu)}
# Note: no thermal rate needed in e2
branch_data = {
    1: {'from': 1, 'to': 2, 'x': 0.1},
    2: {'from': 1, 'to': 3, 'x': 0.1},
    3: {'from': 2, 'to': 3, 'x': 0.2},
}

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build Pyomo Model
# ─────────────────────────────────────────────────────────────────
model = ConcreteModel()

# Sets
model.BUS    = Set(initialize=buses)
model.GEN    = Set(initialize=gen_data.keys())
model.BRANCH = Set(initialize=branch_data.keys())

# Parameters
model.busLoad       = Param(model.BUS,    initialize=load_data)
model.gen_bus       = Param(model.GEN,    initialize={g: gen_data[g]['bus']   for g in gen_data}, within=model.BUS)
model.gen_min       = Param(model.GEN,    initialize={g: gen_data[g]['Pgmin'] for g in gen_data})
model.gen_max       = Param(model.GEN,    initialize={g: gen_data[g]['Pgmax'] for g in gen_data})
model.gen_OpCost    = Param(model.GEN,    initialize={g: gen_data[g]['c']     for g in gen_data})
model.branch_frmbus = Param(model.BRANCH, initialize={k: branch_data[k]['from'] for k in branch_data}, within=model.BUS)
model.branch_tobus  = Param(model.BRANCH, initialize={k: branch_data[k]['to']   for k in branch_data}, within=model.BUS)
model.branch_x      = Param(model.BRANCH, initialize={k: branch_data[k]['x']    for k in branch_data})

# Decision Variables
model.G     = Var(model.GEN)
model.pk    = Var(model.BRANCH)
model.theta = Var(model.BUS)

# ── Objective ────────────────────────────────────────────────────
def obj_rule(model):
    return sum(model.gen_OpCost[g] * model.G[g] for g in model.GEN)
model.obj = Objective(rule=obj_rule, sense=minimize)

# ── Constraints ──────────────────────────────────────────────────
# DC line flow equations (no branch limits in e2)
def lineFlowEqs_rule(model, k):
    return model.pk[k] / BaseMW == (
        model.theta[model.branch_frmbus[k]] - model.theta[model.branch_tobus[k]]
    ) / model.branch_x[k]
model.lineFlowEqs = Constraint(model.BRANCH, rule=lineFlowEqs_rule)

# Generator capacity limits
def genLimits_rule(model, g):
    return inequality(model.gen_min[g], model.G[g], model.gen_max[g])
model.genLimits = Constraint(model.GEN, rule=genLimits_rule)

# Nodal power balance
def NodalPowerBalance_rule(model, n):
    gen_power = sum(model.G[g]  for g in model.GEN    if model.gen_bus[g]       == n)
    power_in  = sum(model.pk[k] for k in model.BRANCH if model.branch_tobus[k]  == n)
    power_out = sum(model.pk[k] for k in model.BRANCH if model.branch_frmbus[k] == n)
    return gen_power + power_in - power_out == model.busLoad[n]
model.NodalPowerBalance = Constraint(model.BUS, rule=NodalPowerBalance_rule)

# Fix reference bus angle
model.theta[refBus].fix(0)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Solve
# ─────────────────────────────────────────────────────────────────
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

start_time = time.time()
results = solver.solve(model, tee=True)
solve_time = time.time() - start_time

# ─────────────────────────────────────────────────────────────────
# Display Results
# ─────────────────────────────────────────────────────────────────
print('\n=== Generator Dispatch ===')
for g in model.GEN:
    print(f'  G[{g}] = {model.G[g].value:.4f} MW')

print('\n=== Branch Flows ===')
for k in model.BRANCH:
    print(f'  pk[{k}] (Bus {branch_data[k]["from"]}→{branch_data[k]["to"]}) = {model.pk[k].value:.4f} MW')

print('\n=== Voltage Angles ===')
for n in model.BUS:
    print(f'  theta[{n}] = {model.theta[n].value:.6f} rad')

total_cost = sum(gen_data[g]['c'] * model.G[g].value for g in model.GEN)
print(f'\n=== Total Cost = ${total_cost:.2f} ===')
print(f'Total solve elapsed time: {solve_time:.4f} seconds')